# Gradient boosting / XGBoost (eXtreme Gradient Boosting)

_Classical ML_

**Build the model in stages: each new tree fixes the last one's mistakes.**

Random forests build many trees in parallel and vote. Gradient boosting builds them one at a time, in sequence.

     Each new tree is trained to fix what the current model still gets wrong — its residual errors.

---

This notebook is a practice scaffold for the **Gradient boosting / XGBoost (eXtreme Gradient Boosting)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

In [ ]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

X, y = make_regression(n_samples=400, n_features=6, noise=10.0, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)

gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1,
                               max_depth=3, random_state=0)
gb.fit(Xtr, ytr)

pred = gb.predict(Xte)
print("n_estimators:", gb.n_estimators_)
print("test R^2:", round(r2_score(yte, pred), 3))
print("test MSE:", round(mean_squared_error(yte, pred), 2))

# error after 1, 50, 200 trees -> shrinks as stages accumulate
for i, yp in enumerate(gb.staged_predict(Xte), start=1):
    if i in (1, 50, 200):
        print("after %3d trees  test MSE = %.2f" % (i, mean_squared_error(yte, yp)))

## Visualize the data & results

_On the Breast Cancer data, does held-out error keep shrinking as boosting rounds accumulate?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import log_loss

bc = load_breast_cancer()
Xtr, Xte, ytr, yte = train_test_split(
    bc.data, bc.target, test_size=0.25, random_state=0,
    stratify=bc.target)

gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                max_depth=3, random_state=0).fit(Xtr, ytr)

# test log-loss after each boosting round via staged_predict_proba
rounds = np.arange(1, gb.n_estimators_ + 1)
loss = np.array([log_loss(yte, p)
                 for p in gb.staged_predict_proba(Xte)])

plt.plot(rounds, loss, c="#4ea1ff")
plt.title("Gradient boosting on Breast Cancer: test log-loss per round")
plt.xlabel("number of trees")
plt.ylabel("test log-loss")
plt.show()